# Hierarchischer Ansatz: Tuning und Vorverarbeitung

**Autor:** jd · **Datum:** 2026-10-12

**Ziel:** Die hierarchische RBF-SVM durch ein erweitertes Raster und einzelne Vorverarbeitungs-Varianten verbessern (#89) – Entscheidung nur per CV, eine Bewertung am Ende.

In [ ]:
%load_ext autoreload
%autoreload 2

import ast

import pandas as pd
from mlflow import MlflowClient
from sklearn.metrics import balanced_accuracy_score

from awp2.config import CROP_COL, MLFLOW_TRACKING_URI, MODEL_DOCS_FIGURES_DIR, STAGE_COL
from awp2.data import load_split
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models import make_hierarchical_svm
from awp2.plots import plot_cv_scores, plot_tuning_candidates, save_doc_figure
from awp2.preprocessing import PreprocessingConfig

In [ ]:
split = load_split()
split.X_train.shape

## Ausgangspunkt

`hierarchical_svm` (#87): RBF-SVM → RBF-SVM, standardisierte Spektren, AEZ one-hot, Monat
zyklisch; CV 0.839, Validierung 0.844. Das Stadium-Optimum (`C=100`, `gamma=0.003`) lag am Rand
des Rasters. Jeder Schritt hier entscheidet nur per CV auf dem Trainingsteil; die Validierung
wird am Ende genau einmal für den Gesamtsieger genutzt.

In [ ]:
HARD_SEARCH_RUN_ID = "2b851b8899c64828995dee02d90a5299"  # hierarchical_svm_search, #87
hard_search = MlflowClient(MLFLOW_TRACKING_URI).get_run(HARD_SEARCH_RUN_ID).data
reference_cv_score = hard_search.metrics["cv_bacc_combined_mean"]
reference_params = {
    name.removeprefix("best."): ast.literal_eval(value)
    for name, value in hard_search.params.items()
    if name.startswith("best.")
}
base_preprocessing = PreprocessingConfig(use_spectral_standard_scale=True)
# a CV gain below this is within the fold-to-fold noise of the reference (std 0.010)
MIN_CV_GAIN = 0.005
TOP_CANDIDATES = 15
reference_cv_score, reference_params

## 1. Erweitertes SVM-Raster

Stadium-`C` nach oben und `gamma` nach unten über den Rand hinaus; die Kultur bleibt um ihr
inneres Optimum.

In [ ]:
wide = tune(
    make_hierarchical_svm(),
    TuneConfig(
        name="hierarchical_svm_wide_search",
        approach="hierarchical",
        description="Erweitertes Raster für RBF-SVM → RBF-SVM: Stadium-C nach oben, "
        "Stadium-gamma nach unten über den Rand von hierarchical_svm_search hinaus.",
        preprocessing=base_preprocessing,
        param_grid={
            "crop_estimator__C": [30.0, 100.0],
            "crop_estimator__gamma": [0.003, 0.01],
            "stage_estimator__C": [100.0, 300.0, 1000.0],
            "stage_estimator__gamma": [0.0003, 0.001, 0.003],
        },
    ),
)
wide_best = wide.candidates.loc[wide.candidates["rank"].idxmin()]
wide_reference = wide.candidates.loc[
    [params == reference_params for params in wide.candidates["params"]]
].iloc[0]
fold_gain = (wide_best.filter(like="fold_") - wide_reference.filter(like="fold_")).astype(float)
(
    wide.best_params,
    wide_best[["mean", "std"]],
    wide_reference[["mean", "std"]],
    fold_gain.round(4),
)

In [ ]:
wide_ax = plot_tuning_candidates(
    wide.candidates.nsmallest(TOP_CANDIDATES, "rank"), color_by="stage_estimator__C"
)
wide_ax.axvline(reference_cv_score, color="black", linestyle="--", linewidth=1)
wide_ax.set_xlim(0.78, 0.87)
wide_ax.set_title(f"Erweitertes Raster: beste {TOP_CANDIDATES} CV-Kandidaten")
save_doc_figure(wide_ax.figure, "hierarchical_wide_tuning", MODEL_DOCS_FIGURES_DIR)

## 2. Vorverarbeitung: je genau eine Änderung

Basis ist die beste Einstellung aus Schritt 1. Jede Variante ändert genau ein Feld der
`PreprocessingConfig` (Werte für Savitzky-Golay nach Pascals Pipeline-Test, PCA um dessen 13–15
Komponenten). `gamma` wird je Stufe zwischen dem bisherigen Bestwert und `"scale"` gewählt, weil
sich mit der Merkmalszahl (PCA, Indizes) die sinnvolle Kernelbreite ändert. Vegetationsindizes
laufen ohne SNV (#91). Der numerische Monat braucht das veraltete `scale=True`, damit Month
standardisiert wird – genau die Einstellung der Referenz `combined_svm_expanded`.

In [ ]:
variants = {
    "SNV": {"use_spectral_snv": True},
    "Savitzky-Golay (5, 2)": {"use_savgol": True, "savgol_window_length": 5, "savgol_polyorder": 2},
    "PCA 15": {"spectral_pca_components": 15},
    "PCA 30": {"spectral_pca_components": 30},
    "PCA 60": {"spectral_pca_components": 60},
    "Vegetationsindizes": {"use_vegetation_indices": True},
    "Monat numerisch": {
        "use_spectral_standard_scale": False,
        "scale": True,
        "use_cyclic_month": False,
    },
}


def variant_config(changes: dict) -> PreprocessingConfig:
    return PreprocessingConfig(**{**base_preprocessing.model_dump(), **changes})


base_model = make_hierarchical_svm().set_params(**wide.best_params)
gamma_grid = {
    "crop_estimator__gamma": [wide.best_params["crop_estimator__gamma"], "scale"],
    "stage_estimator__gamma": [wide.best_params["stage_estimator__gamma"], "scale"],
}
ablations = {
    label: tune(
        base_model,
        TuneConfig(
            name=f"hierarchical_prep_{index}_search",
            approach="hierarchical",
            description=f"Vorverarbeitungs-Variante „{label}“ gegenüber der Basis "
            "(standardisierte Spektren, zyklischer Monat); gamma bisher vs. scale je Stufe.",
            preprocessing=variant_config(changes),
            param_grid=gamma_grid,
        ),
    )
    for index, (label, changes) in enumerate(variants.items())
}

In [ ]:
cv_table = pd.DataFrame(
    {
        label: {
            "CV": tuned.best_score,
            "Gewinn": tuned.best_score - wide.best_score,
            "params": tuned.best_params,
            "run": tuned.run_id[:8],
        }
        for label, tuned in {"Basis (Schritt 1)": wide, **ablations}.items()
    }
).T.sort_values("CV", ascending=False)
cv_table

In [ ]:
prep_ax = plot_cv_scores(
    {"Basis (Schritt 1)": wide.candidates}
    | {label: tuned.candidates for label, tuned in ablations.items()},
    reference=wide.best_score,
)
prep_ax.set_title("Vorverarbeitung: je eine Änderung gegenüber der Basis")
save_doc_figure(prep_ax.figure, "hierarchical_preprocessing", MODEL_DOCS_FIGURES_DIR)

## 3. Kombinieren und Gesamtsieger

Kombiniert werden nur Varianten, die die Basis um mindestens `MIN_CV_GAIN` schlagen; von den
PCA-Varianten höchstens die beste, weil sie sich ausschließen.

In [ ]:
winners = [
    label
    for label, tuned in sorted(ablations.items(), key=lambda item: -item[1].best_score)
    if tuned.best_score - wide.best_score >= MIN_CV_GAIN
]
pca_winners = [label for label in winners if label.startswith("PCA")]
winners = [label for label in winners if not label.startswith("PCA")] + pca_winners[:1]

combined = None
if len(winners) >= 2:
    combined_changes = {key: value for label in winners for key, value in variants[label].items()}
    combined = tune(
        base_model,
        TuneConfig(
            name="hierarchical_prep_combined_search",
            approach="hierarchical",
            description=f"Kombination der CV-Gewinner {winners}.",
            preprocessing=variant_config(combined_changes),
            param_grid=gamma_grid,
        ),
    )
winners, combined.best_score if combined else None

In [ ]:
contenders = {"Basis (Schritt 1)": (wide, base_preprocessing)}
contenders |= {label: (ablations[label], variant_config(variants[label])) for label in winners}
if combined:
    contenders["Kombination"] = (combined, variant_config(combined_changes))
best_label, (best_tuned, best_preprocessing) = max(
    contenders.items(), key=lambda item: item[1][0].best_score
)
is_reference = best_preprocessing == base_preprocessing and best_tuned.best_params == {
    name: reference_params[name] for name in best_tuned.best_params
}
best_label, round(best_tuned.best_score, 4), is_reference

In [ ]:
if is_reference:
    final = {"Entscheidung": "Referenz hierarchical_svm (#87) bleibt; kein neuer Lauf"}
else:
    result = run(
        best_tuned.best_model,
        RunConfig(
            name="hierarchical_svm_tuned",
            approach="hierarchical",
            description=f"Gesamtsieger aus #89 ({best_label}), einmal bewertet.",
            preprocessing=best_preprocessing,
            tuning_run=best_tuned.run_id,
        ),
    )
    final = result.metrics.model_dump()
final

## 4. Schwache Kulturen

Stadium-BAcc je Kultur auf den Zeilen mit richtig erkannter Kultur (out-of-fold, nur
Trainingsteil) für das Referenzmodell `hierarchical_svm`. Ein deutlich schwächeres
Stadium-Modell einer Kultur wäre ein Grund für eigene Hyperparameter je Kultur.

In [ ]:
reference_model = make_hierarchical_svm().set_params(**reference_params)
oof = out_of_fold_predictions(reference_model, wide.config)
crop_correct = oof[CROP_COL] == split.y_train[CROP_COL]
stage_by_crop = pd.DataFrame(
    {
        crop: {
            "Stadium-BAcc": balanced_accuracy_score(
                split.y_train.loc[rows, STAGE_COL], oof.loc[rows, STAGE_COL]
            ),
            "Zeilen": rows.sum(),
            "seltenstes Stadium": split.y_train.loc[rows, STAGE_COL].value_counts().idxmin(),
            "dessen Zeilen": split.y_train.loc[rows, STAGE_COL].value_counts().min(),
        }
        for crop in sorted(split.y_train[CROP_COL].unique())
        for rows in [crop_correct & (split.y_train[CROP_COL] == crop)]
    }
).T
stage_by_crop

In [ ]:
true_pairs = split.y_train.loc[crop_correct]
recall_by_pair = (
    (oof.loc[crop_correct, STAGE_COL] == true_pairs[STAGE_COL])
    .groupby([true_pairs[CROP_COL], true_pairs[STAGE_COL]])
    .agg(["mean", "size"])
    .sort_values("mean")
)
recall_by_pair.head(8).round(3)

## Befund

- Erweitertes Raster: Stadium `C=1000` gewinnt nur +0.001 – unter `MIN_CV_GAIN`, die
  Fold-Differenzen zur Referenz wechseln das Vorzeichen (−0.025, −0.004, +0.007, +0.011,
  +0.016). Das Optimum liegt wieder am Rasterrand, die Scores sind aber flach (C=300: 0.839) –
  ein Plateau. Die Schwelle fehlte in Schritt 1, daher lief `run()` dennoch (0.839);
  `hierarchical_svm` bleibt Referenz, begründet durch die CV.
- Die Ablationen liefen auf Stadium `C=1000` (Basis 0.840) mit je 4 statt 36 Kandidaten – der
  Vergleich begünstigt eher die Basis. Keine Variante gewinnt ≥ 0.005; SNV (−0.018, 4 von 5
  Folds schlechter) und PCA 15 (−0.009) schaden. Zyklischer und numerischer Monat sind
  gleichwertig.
- Nach der Kulturstufe sind die Stadium-Modelle von Baumwolle (0.84; Mature_Senesc 0.75,
  Emerge_VEarly 0.82) und Soja (0.93) die Schwachstelle – Reis ist trotz 66 Zeilen perfekt.